---
Title: Data Validation Model
author:
  - name: "edesz"
    orcid: "0000-0001-9039-5496"
    attributes:
      github: "edesz"
date: 2026-05-01
---

# Validate Raw Data

## About

Having completed the task of identifying the cohort of at-risk customers that maximizes Return on Investment (ROI), we now need to ensure new customer data has the same characteristics as the sample of data that was provided to us by the client. This is necessary to ensure we our understanding of the bank's new (unseen) credit card customers' characteristics has not changed. By doing this, we ensure the analysis we developed using the historical data can also be applied to the new customers data and the client can realize the estimated ROI from the best cohort of at-risk customers. Any *data drift* can lead the client targeting the wrong customers or miss those truly at risk, which would waste marketing resources and not achieve our estimated ROI.

With this in mind, in this step, we develop a customer schema to validate raw credit card customer data. The schema is used to validate the raw historical data that the client gave us for which the outcome of churn is known. The same schema can be used to validate new data as well, with no changes.

[Pandera](https://pypi.org/project/pandera/) provides a robust framework to implement this validation. It allows for defining a formal schema that checks for correct data types, value ranges, and specific categories for every customer attribute. By using Pandera, we can automatically catch the customers that do not have the same raw characteristics of the customers in the historical data. It can also check for missing values since we did not have any in the historical customer data.

In summary, this step ensures the new credit card dataset has the same characteristics of the historical sample of data that we used to identify at-risk customers and to estimate cohort size and predicted ROI.

If this step runs successfully on the client's customer data then the data has been verified to meet the requirements of the schema defined here. If the step does not run successfully then the data does not conform to this expected schema. This could indicate an error in data retrieval (new columns, missing data, different data types, etc.) or that customer behaviour (numerical features) or attributes (categorical or ordinal features) have changed relative to our expectations based on this schema.

:::{.callout-note}
### Output

None
:::

## Python Imports

The required Python modules are imported below

In [1]:
# | code-fold: true
import os
import sys
from pathlib import Path

import boto3
import pandas as pd
import pandera as pa
from dotenv import load_dotenv
from great_tables import GT, loc, md, style

Define the path to the project root directory

In [2]:
PROJ_ROOT = Path.cwd().parent

Load environment variables with secrets for use in `boto3`

In [3]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

Import the custom Python modules that the team has developed

In [4]:
import r2.io_utils as r2io
from utils.df_utils import show_df
from utils.display_utils import markdown_highlight_filtered, pygments_highlight

In [5]:
data_models_path = str(PROJ_ROOT)
if data_models_path not in sys.path:
    sys.path.append(data_models_path)

Import the custom data schema for performing the data validation

In [6]:
from data_models.staging import CreditCardCustomerSchema

## User Inputs

Below we define variables that will be used later

In [7]:
# name of raw data key (file) in private R2 bucket
r2_key_raw_data = "BankChurners.xlsx"

Define an authenticated `boto3` S3 client to interact with the Cloudflare R2 bucket

In [8]:
account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
bucket_name = os.getenv("BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

## Extract

We will first load all raw customer data from the R2 bucket

In [9]:
df = r2io.pandas_read_xlsx_r2(s3_client, bucket_name, r2_key_raw_data, {})
print(f"Loaded {len(df):,} rows of raw data")
_ = show_df(df)

Loaded 10,127 rows of raw data
Shape: 10,127 rows X 21 columns, Memory Usage: 1956.885 KB


This is shown below

In [10]:
# | echo: false
gt = (
    GT(df.head(1))
    .tab_header(md("**First Row of Raw Customer Data**"))
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["CLIENTNUM"]),
    )
    .tab_style(
        style=[style.fill(color="red"), style.text(color="white")],
        locations=loc.body(columns=["Attrition_Flag"]),
    )
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
)
gt

GT(_tbl_data=   CLIENTNUM     Attrition_Flag  Customer_Age Gender  Dependent_count  \
0  768805383  Existing Customer            45      M                3   

  Education_Level Marital_Status Income_Category Card_Category  \
0     High School        Married     $60K - $80K          Blue   

   Months_on_book (Length of relationship with bansk[months])  ...  \
0                                                 39           ...   

   Months_Inactive_12_mon (Card not used)  \
0                                       1   

   Contacts_Count_12_mon (Number of contacts in 12 months)  Credit_Limit  \
0                                                  3             12691.0   

   Total_Revolving_Bal (Balance unpaid at month end)  \
0                                                777   

   Avg_Open_To_Buy (Difference between the credit limit and the balance)  \
0                                            11914.0                       

   Total_Amt_Chng_Q4_Q1(Ratio Q4/Q1)  \
0                              1.335   

   Total_Trans_Amt ( Total Transactions Value 12 months)  \
0                                               1144       

   Total_Trans_Ct (Transaction Count 12 months)  \
0                                            42   

   Total_Ct_Chng_Q4_Q1 (Change in the transaction amount Q4/Q1)  \
0                                              1.625              

   Avg_Utilization_Ratio (Credit usage/Total Credit available)  
0                                              0.061            

[1 rows x 21 columns], _body=<great_tables._gt_data.Body object at 0x766a6eb49640>, _boxhead=Boxhead([ColInfo(var='CLIENTNUM', type=<ColInfoTypeEnum.default: 1>, column_label='CLIENTNUM', column_align='right', column_width=None), ColInfo(var='Attrition_Flag', type=<ColInfoTypeEnum.default: 1>, column_label='Attrition_Flag', column_align='center', column_width=None), ColInfo(var='Customer_Age', type=<ColInfoTypeEnum.default: 1>, column_label='Customer_Age', column_align='right', column_width=None), ColInfo(var='Gender', type=<ColInfoTypeEnum.default: 1>, column_label='Gender', column_align='center', column_width=None), ColInfo(var='Dependent_count', type=<ColInfoTypeEnum.default: 1>, column_label='Dependent_count', column_align='right', column_width=None), ColInfo(var='Education_Level', type=<ColInfoTypeEnum.default: 1>, column_label='Education_Level', column_align='center', column_width=None), ColInfo(var='Marital_Status', type=<ColInfoTypeEnum.default: 1>, column_label='Marital_Status', column_align='center', column_width=None), ColInfo(var='Income_Category', type=<ColInfoTypeEnum.default: 1>, column_label='Income_Category', column_align='center', column_width=None), ColInfo(var='Card_Category', type=<ColInfoTypeEnum.default: 1>, column_label='Card_Category', column_align='center', column_width=None), ColInfo(var='Months_on_book (Length of relationship with bansk[months])', type=<ColInfoTypeEnum.default: 1>, column_label='Months_on_book (Length of relationship with bansk[months])', column_align='right', column_width=None), ColInfo(var='Total_Relationship_Count (How many products with customer) ', type=<ColInfoTypeEnum.default: 1>, column_label='Total_Relationship_Count (How many products with customer) ', column_align='right', column_width=None), ColInfo(var='Months_Inactive_12_mon (Card not used)', type=<ColInfoTypeEnum.default: 1>, column_label='Months_Inactive_12_mon (Card not used)', column_align='right', column_width=None), ColInfo(var='Contacts_Count_12_mon (Number of contacts in 12 months)', type=<ColInfoTypeEnum.default: 1>, column_label='Contacts_Count_12_mon (Number of contacts in 12 months)', column_align='right', column_width=None), ColInfo(var='Credit_Limit', type=<ColInfoTypeEnum.default: 1>, column_label='Credit_Limit', column_align='right', column_width=None), ColInfo(var='Total_Revolving_Bal (Balance unpaid at month end)', type=<ColInfoTypeEnum.default: 1>, column_label='Total_Revolving_Bal (Balance unpaid at month end)', c

## Data Validatation

In this section data validation checks performed on all columns in the raw data using the [`pandera` Python package](https://pypi.org/project/pandera/).

Three types of checks are performed

1. at the **column level** on individual columns
   - for numerical columns
     - to ensure their datatype and bounds (minimum and/or maximum values) are expected from the sample data
     - to ensure their Z-scores are expected from the sample data
   - for categorical and ordinal columns
     - to ensure their unique values are the same as those from the sample data
   - for the customer identifier column
     - to ensure the identifier is unique to each customer
2. at the **dataset level** to ensure inter-column relationships are in line with those expected from the sample data
3. at the **observation (row) level** on individual customers based on their credit card activity

### Checks

#### Column-Level Checks - Part 1/2 (Datatypes, Bounds, Expected Values)

**Categorical and Ordinal Columns (expected values using `isin`)**

1. Ensures values belong to a predefined set
2. Prevents unexpected categories due to data drift or ingestion errors

Below is the section of the data schema that performs these checks on the categorical and ordinal columns

In [11]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "base_schema.py"),
    unwanted_ranges=[(0, 11), (13, 21), (58, 126)],
)

```python
class BankChurnersBase(pa.DataFrameModel):
    """Structural definition of the BankChurners dataset."""

    # Categoricals
    Gender: Series[pd.CategoricalDtype] = Field(isin=["M", "F"])

    Marital_Status: Series[pd.CategoricalDtype] = Field(
        isin=["Married", "Single", "Divorced", "Unknown"]
    )

    Card_Category: Series[pd.CategoricalDtype] = Field(
        isin=["Blue", "Silver", "Gold", "Platinum"]
    )

    Dependent_count: Series[int] = Field(ge=0, le=5)

    # Ordinals
    Education_Level: Series[pd.CategoricalDtype] = Field(
        isin=[
            "Uneducated",
            "High School",
            "College",
            "Graduate",
            "Post-Graduate",
            "Doctorate",
            "Unknown",
        ]
    )

    Income_Category: Series[pd.CategoricalDtype] = Field(
        isin=[
            "Less than $40K",
            "$40K - $60K",
            "$60K - $80K",
            "$80K - $120K",
            "$120K +",
            "Unknown",
        ]
    )
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "base_schema.py"),
    unwanted_lines=(
        list(range(0, 11)) + list(range(13, 21)) + list(range(58, 126))
    ),
    style="vs",
)

**Numerical Columns (bounds using `ge`, `le`, `gt`)**

1. Enforce realistic domain limits (e.g., age, months, ratios)
2. Prevent invalid or corrupted values (e.g., negative balances)

Below is the section of the data schema that performs these checks on all numerical columns

In [13]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "base_schema.py"),
    unwanted_ranges=[(0, 11), (14,), (15, 59), (121, 126)],
)

```python
class BankChurnersBase(pa.DataFrameModel):
    """Structural definition of the BankChurners dataset."""

    # Numerical columns
    Customer_Age: Series[int] = Field(ge=18, le=100)

    Months_on_book: Series[int] = Field(
        alias="Months_on_book (Length of relationship with bansk[months])",
        ge=0,
        le=600,
    )

    Total_Relationship_Count: Series[int] = Field(
        alias="Total_Relationship_Count (How many products with customer) ",
        ge=1,
        le=10,
    )

    Months_Inactive_12_mon: Series[int] = Field(
        alias="Months_Inactive_12_mon (Card not used)", ge=0, le=12
    )

    Contacts_Count_12_mon: Series[int] = Field(
        alias="Contacts_Count_12_mon (Number of contacts in 12 months)",
        ge=0,
        le=12,
    )

    Credit_Limit: Series[float] = Field(gt=0)

    Total_Revolving_Bal: Series[float] = Field(
        alias="Total_Revolving_Bal (Balance unpaid at month end)", ge=0
    )

    Avg_Open_To_Buy: Series[float] = Field(
        alias=(
            "Avg_Open_To_Buy (Difference between the credit limit and the "
            "balance)"
        ),
        ge=0,
    )

    Total_Amt_Chng_Q4_Q1: Series[float] = Field(
        alias="Total_Amt_Chng_Q4_Q1(Ratio Q4/Q1)", ge=0
    )

    Total_Trans_Amt: Series[float] = Field(
        alias="Total_Trans_Amt ( Total Transactions Value 12 months)", ge=0
    )

    Total_Trans_Ct: Series[int] = Field(
        alias="Total_Trans_Ct (Transaction Count 12 months)", ge=0
    )

    Total_Ct_Chng_Q4_Q1: Series[float] = Field(
        alias="Total_Ct_Chng_Q4_Q1 (Change in the transaction amount Q4/Q1)",
        ge=0,
    )

    Avg_Utilization_Ratio: Series[float] = Field(
        alias="Avg_Utilization_Ratio (Credit usage/Total Credit available)",
        ge=0,
        le=1,
    )
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "base_schema.py"),
    unwanted_lines=(
        list(range(0, 11)) + [14] + list(range(15, 59)) + list(range(121, 126))
    ),
    style="vs",
)

**Identifier Column `CLIENTNUM` (uniqueness)**

1. Ensures each customer is uniquely identified
2. Prevents duplicate records

Below is the section of the data schema that performs these checks on the identifier column

In [15]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "base_schema.py"),
    unwanted_ranges=[(0, 11), (13, 14), (17, 126)],
)

```python
class BankChurnersBase(pa.DataFrameModel):
    """Structural definition of the BankChurners dataset."""
    # Identifier
    CLIENTNUM: Series[int] = pa.Field(unique=True)
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "base_schema.py"),
    unwanted_lines=(
        list(range(0, 11)) + list(range(13, 14)) + list(range(17, 126))
    ),
    style="vs",
)

#### Dataset-Level Checks

Next, we focus on the relationships between columns in the raw dataset. We expect the following relationships

**Credit Balance**

$$
\text{Credit\_Limit} \sim \text{Total\_Revolving\_Bal} + \text{Avg\_Open\_To\_Buy}
$$

Below is the section of the data schema that performs this test

In [17]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_ranges=[(0, 11), (20, 28), (38, 322)],
)

```python
class BehaviouralChecks:
    """Reusable checks for bank customer behavior."""

    # cross-field validation
    @pa.dataframe_check
    def check_credit_balance_consistency(cls, df: pd.DataFrame) -> pd.Series:
        """
        Validate that credit components sum correctly.

        """
        col_1 = "Total_Revolving_Bal (Balance unpaid at month end)"
        col_2 = (
            "Avg_Open_To_Buy (Difference between the credit limit and the "
            "balance)"
        )
        lhs = df["Credit_Limit"]
        rhs = df[col_1].add(df[col_2])
        return (lhs == rhs).all()
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_lines=(
        list(range(0, 11)) + list(range(20, 28)) + list(range(38, 322))
    ),
    style="vs",
)

**Utilization Ratio**

$$
\text{Avg\_Utilization\_Ratio} = \frac{\text{Total\_Revolving\_Bal}}{\text{Credit\_Limit}}
$$

Below is the section of the data schema that performs this test

In [19]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_ranges=[(0, 38), (43, 51), (70, 322)],
)

```python
@pa.dataframe_check
def check_utilization_ratio(cls, df: pd.DataFrame) -> pd.Series:
    """
    Validate utilization ratio consistency.

    """
    numerator_c = "Total_Revolving_Bal (Balance unpaid at month end)"
    denominator_c = "Credit_Limit"
    true_c = "Avg_Utilization_Ratio (Credit usage/Total Credit available)"
    expected = df[numerator_c].div(df[denominator_c]).rename("expected")
    true = df[true_c].rename("true")
    num_decimals = 3
    return (
        pd.concat([true, expected], axis=1)
        .assign(
            check=lambda df: (
                df["true"]
                .round(num_decimals)
                .equals(df["expected"].round(num_decimals))
            )
        )["check"]
        .all()
    )
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_lines=(
        list(range(0, 38)) + list(range(43, 51)) + list(range(70, 322))
    ),
    style="vs",
)

#### Anomaly Detection Based on Credit Card Customer Behaviour

Next, based on the data provided to us by the client, we do not expect customers with the following criteria in the dataset

1. High utilization (≥ 0.8) and low activity
2. Transaction count ≤ 10
3. Transaction amount ≤ 1000

Any such customers should be classified as an anomaly. As an example, we want to detect if customers are present with  a maxed-out their credit limit but with little credit card usage, which is an unrealistic scenario. So a validation check is implemented to capture such risky or unusual patterns in customer behavior.

Below is the implementation of this test

In [21]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_ranges=[(0, 116), (120, 130), (143, 322)],
)

```python
def check_behavioral_anomalies(cls, df: pd.DataFrame) -> pd.Series:
    """
    Detect behavioral anomalies based on credit usage patterns.

    """
    c1 = "Avg_Utilization_Ratio (Credit usage/Total Credit available)"
    c2 = "Total_Trans_Ct (Transaction Count 12 months)"
    c3 = "Total_Amt_Chng_Q4_Q1(Ratio Q4/Q1)"
    high_util = df[c1] >= 0.8
    low_txn_count = df[c2] <= 10
    low_txn_amt = df[c3] <= 1000

    anomaly_mask = high_util & low_txn_count & low_txn_amt

    # Valid rows = NOT anomalous
    return ~anomaly_mask
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_lines=(
        list(range(0, 116)) + list(range(120, 130)) + list(range(143, 322))
    ),
    style="vs",
)

### Column-Level Checks - Part 2/2 (Z-Score to Detect Outliers)

Finally, we use z-score tests to check for outliers in numerical columns.

Based on the sample data provided to us by the client, we expect the following

**|Z-Score| ≤ 4.0**

The most common z-score threshold is 3 (or -3), meaning data points (rows) further than 3 standard deviations from the mean will fail this test. Based on our EDA of the sample of the data provided by the client, we have outliers in the sample data, so we are allowing extreme outliers by accepting a z-score of up to 4.

These checks identify extreme values across the numerical columns, which helps detect

1. data entry errors
2. rare but suspicious customer records
3. anomalies in the numerical variable's distribution

A row (customer) fails this test if any monitored column is an outlier. We are using the population standard deviation, which uses a `ddof=0`, which handles zero-variance columns safely.

The following test was implemented to capture this logic

In [23]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_ranges=[(0, 71), (75, 83), (115, 322)],
)

```python
def check_zscore_outliers(cls, df: pd.DataFrame) -> pd.Series:
    """
    Detect statistical outliers using z-score bounds.

    """
    cols = [
        "Customer_Age",
        "Months_on_book (Length of relationship with bansk[months])",
        "Total_Relationship_Count (How many products with customer) ",
        "Months_Inactive_12_mon (Card not used)",
        "Contacts_Count_12_mon (Number of contacts in 12 months)",
        "Credit_Limit",
        "Total_Revolving_Bal (Balance unpaid at month end)",
        (
            "Avg_Open_To_Buy (Difference between the credit limit and "
            "the balance)"
        ),
        "Total_Trans_Ct (Transaction Count 12 months)",
    ]

    z_thresh = 4.0

    sub_df = df[cols]

    # Compute z-scores safely
    mean = sub_df.mean()
    std = sub_df.std(ddof=0).replace(0, np.nan)

    z_scores = (sub_df - mean) / std

    # Identify rows where ANY column exceeds threshold
    outlier_mask = (np.abs(z_scores) > z_thresh).any(axis=1)

    # Valid rows = NOT outliers
    return ~outlier_mask
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_lines=(
        list(range(0, 71)) + list(range(75, 83)) + list(range(115, 322))
    ),
    style="vs",
)

### Property-Based Tests

`pandera` has `Hypothesis` testing which allows for running statistical tests like `two_sample_ttest` to validate data relationships by checking for distributions or differences between data groups. This allows us to define tests beyond simple value checks. Here, seven such tests were used as part of the data schema.

#### Unpaid Credit Card Balance (Month End)

Below we test a hypothesis that the average unpaid credit card balance at month end is significantly different for existing and churned credit card customers. If the hypothesis is proven and the test passes then this would confirm our intuition of different behaviour between these two groups of customers

In [25]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_ranges=[(0, 143), (152, 162), (177, 302)],
)

```python
@pa.check(
    "Total_Revolving_Bal (Balance unpaid at month end)",
    groupby="Attrition_Flag",
    name="revolving_bal_t_test",
)
def check_balance_stats(cls, grouped_data):
    """
    Validate statistical difference in revolving balances.

    """
    # Access the groups directly from the mapping
    existing = grouped_data.get("Existing Customer")
    attrited = grouped_data.get("Attrited Customer")

    # Guard clause: ensure both groups exist in the data batch
    if existing is None or attrited is None:
        return False

    # Perform the t-test to check if the means are different
    _, p_value = stats.ttest_ind(existing, attrited, equal_var=True)

    # Check if the difference is statistically significant (p < 0.05)
    return p_value < 0.05
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_lines=(
        list(range(0, 143)) + list(range(152, 162)) + list(range(177, 302))
    ),
    style="vs",
)

:::{.callout-note}
## Choice of Test Depends on Population Variance

[Welch's t-test](wiki:Welch's_t-test#Assumptions) allows for different population variances. By comparison, the [Student T-test](wiki:Student's_t-test) for the means of two independent (unrelated) samples requires the [two groups to have the same population variance](wiki:Student's_t-test#Uses). Here, we are more restrictive in our test and we require the two groups to have the same population variance. For this reason, we use the Student T-test here with `equal_var=True`.
:::

:::{.callout-tip}
## Variant of T-Test is Used

The Student's T-test for the means of two independent (unrelated) groups of customers is also called the *Two-Sample T-Test*. This is the variant of the T-test that is used to perform the test.
:::

#### Average Transaction Count (Last 12 months)

Below we test a hypothesis that the average total transaction count for existing customers, who we would expect to have a relatively higher number of transactions on their credit card, is in fact significantly higher than that of churned customers

In [27]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_ranges=[(0, 177), (186, 196), (202, 302)],
)

```python
@pa.check(
    "Total_Trans_Ct (Transaction Count 12 months)",
    groupby="Attrition_Flag",
    name="test_transaction_count_drift",
)
def check_trans_count(cls, grouped_data):
    """
    Validate transaction count distribution across customer groups.

    """
    existing = grouped_data.get("Existing Customer")
    attrited = grouped_data.get("Attrited Customer")
    _, p_val = stats.ttest_ind(existing, attrited, equal_var=True)
    # expecting a very high significance here (p < 0.01)
    return p_val < 0.01
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_lines=(
        list(range(0, 177)) + list(range(186, 196)) + list(range(202, 302))
    ),
    style="vs",
)

#### Average Transaction Value (Last 12 months)

Below we adapt the above hypothesis to test that the average total credit card transaction *amount* for existing customers is significantly higher than that of churned customers

In [29]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_ranges=[(0, 203), (212, 223), (229, 302)],
)

```python
@pa.check(
    "Total_Trans_Amt ( Total Transactions Value 12 months)",
    groupby="Attrition_Flag",
    name="test_transaction_amount_drift",
)
def check_trans_amount(cls, grouped_data):
    """
    Validate transaction amount distribution drift.

    """
    existing = grouped_data.get("Existing Customer")
    attrited = grouped_data.get("Attrited Customer")
    _, p_val = stats.ttest_ind(existing, attrited, equal_var=True)
    return p_val < 0.01
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_lines=(
        list(range(0, 203)) + list(range(212, 223)) + list(range(229, 302))
    ),
    style="vs",
)

#### Number of Contacts by the Customer (Last 12 months)

Next, we test a hypothesis that the churned customers contact the bank more often on average during the last 12 months than existing customers

In [31]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_ranges=[(0, 229), (238, 248), (254, 302)],
)

```python
@pa.check(
    "Contacts_Count_12_mon (Number of contacts in 12 months)",
    groupby="Attrition_Flag",
    name="test_contact_frequency",
)
def check_contact_count(cls, grouped_data):
    """
    Validate contact frequency behavioral property.

    """
    existing = grouped_data.get("Existing Customer")
    attrited = grouped_data.get("Attrited Customer")
    _, p_val = stats.ttest_ind(existing, attrited, equal_var=True)
    return p_val < 0.01 and attrited.mean() > existing.mean()
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_lines=(
        list(range(0, 229)) + list(range(238, 248)) + list(range(254, 302))
    ),
    style="vs",
)

#### Average Utilization Ratio

Below we test a hypothesis that the credit utilization ratio of existing customers is higher on average than that of churned customers. Churners stop using their credit card before canceling it and so should have lower utilization on average than existing customers

In [33]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_ranges=[(0, 254), (263, 273), (279, 302)],
)

```python
@pa.check(
    "Avg_Utilization_Ratio (Credit usage/Total Credit available)",
    groupby="Attrition_Flag",
    name="test_utilization_drop",
)
def check_utilization(cls, grouped_data):
    """
    Validate utilization ratio divergence.

    """
    existing = grouped_data.get("Existing Customer")
    attrited = grouped_data.get("Attrited Customer")
    _, p_val = stats.ttest_ind(existing, attrited, equal_var=True)
    return p_val < 0.01 and existing.mean() > attrited.mean()
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_lines=(
        list(range(0, 254)) + list(range(263, 273)) + list(range(279, 302))
    ),
    style="vs",
)

#### Months Inactive During Last 12 Months

Finally, in the last property-based test, we test a hypothesis that churned customers have a significantly higher number of months in the last 12 months during which they have no activity on their credit card than do existing customers

In [35]:
# |echo: false
markdown_highlight_filtered(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_ranges=[(0, 279), (288, 297)],
)

```python
@pa.check(
    "Months_Inactive_12_mon (Card not used)",
    groupby="Attrition_Flag",
    name="test_inactivity_drift",
)
def check_inactivity(cls, grouped_data):
    """
    Validate months of inactivity distribution.

    """
    existing = grouped_data.get("Existing Customer")
    attrited = grouped_data.get("Attrited Customer")
    _, p_val = stats.ttest_ind(existing, attrited, equal_var=True)
    return p_val < 0.01 and attrited.mean() > existing.mean()
```

In [ ]:
# | include: false
pygments_highlight(
    fpath=str(PROJ_ROOT / "data_models" / "staging" / "behavioural_checks.py"),
    unwanted_lines=list(range(0, 279)) + list(range(288, 297)),
    style="vs",
)

### Implementation

Based on the above requirements, the schema is split into three files in `data_models/staging` to keep the Business Logic (Hypotheses) separate from the Data Integrity (Types/Ranges)

1. `base_schema.py`
   - this contains the `pandera` `DataFrameModel` with `pa.Field` definitions only and it catches data corruption and schema changes
   - this is the source of truth for what the columns are and what types they should be
   - this acts as the data dictionary
2. `behavioural_checks.py`
   - this contains the `@pa.check` and `@pa.dataframe_check` logic and statistical tests
   - these are defined as reusable functions and they check the business logic and anomaly detection using Z-scores, T-tests, and financial consistency
3. `schema.py`
   - this inherits from the base schema and adds the checks from `behavioural_checks.py`
   - this is the class that is actually imported for use in validating the raw credit card customer data

The following `pandera` functionality was used to write the validation checks in this schema

1. [`pandera`'s column checks](https://pandera.readthedocs.io/en/stable/dataframe_models.html#basic-usage) are used to perform column-level data validation
2. [`pandera`'s `DataFrame` Checks](https://pandera.readthedocs.io/en/stable/dataframe_models.html#dataframe-checks) are used to implement Dataset-level validation checks, anomaly detection checks and Z-score checks
3. [`pandera`'s hypothesis testing](https://pandera.readthedocs.io/en/stable/hypothesis.html)

This schema is now used to validate the raw data passed to us by the client

In [37]:
try:
    validated_df = CreditCardCustomerSchema.validate(df)
except pa.errors.SchemaError as e:
    print(f"Validation failed: {str(e)}")

## Conclusion

This step has implemented a schema to validate incoming credit card customer data provided to us by the client. Here, we used it to validate the data we already had to validate customers for whom we knew if they churned or did not churn. Following the same approach, this schema can be used to validate new (unseen) data. By doing this, we ensure new customer data demonstrates the same behaviour as that of the data we used to identify at-risk customers. Importantly, we also ensure that our estimated business metrics (e.g. savings) from targeting at-risk customers, which we determined using the provide sample customer data, can also be realized by the client on new data.